In [2]:
import sys
import os
from pathlib import Path

package_path = Path(os.path.abspath("")).parent


In [3]:
from section_identification.preprocess import preprocess_image
from section_identification.filtering import filtering
from section_identification.section_detector import automatic_identification

apply_filtering = True

image1 = package_path / "images/example1.png"
image2 = package_path / "images/example2.png"
image3 = package_path / "images/example3.png"

# the weights can be downloaded from: https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
checkpoint = package_path.parents[0] / "checkpoint/sam_vit_h_4b8939.pth" 

# Write filtering=True to only identify sections
masks = automatic_identification(image1, checkpoint=checkpoint, compress=True, apply_filtering=True)
print(f"Number of masks identified: {len(masks)}")

Compressing the image...
Image shape: (1539, 1566)
Image is compressed.
Loaded cached masks.
Filtering masks...
Most common largest cluster size: 101 out of 123 total masks.
Chosen parameters: eps=100.0, min_samples=1
Filtering completed with chosen parameters: (np.float64(100.0), 1)


2025-02-12 20:39:30.008 python[9120:10676594] +[IMKClient subclass]: chose IMKClient_Modern


interactive(children=(IntSlider(value=0, description='index'), Output()), _dom_classes=('widget-interact',))

Number of masks identified: 101


In [4]:
from section_identification.manual_detector import manual_correction_v2

finalize_func = manual_correction_v2(
    image_path=image1,
    generated_masks=masks,
    checkpoint=checkpoint,   # so we can add new masks
    model_type="vit_h",
    device="cpu", # running on cuda might work
    display_mask_overlay=True
)

[Info] Embedding already exists: /Users/fredericoaraujo/Documents/section_identification/images/example1_embedding.npy


[ WARN:0@34.348] global grfmt_png.cpp:695 read_chunk chunk data is too large


Output()

Output()

: 

In [4]:
from onnx_export import install_and_export_sam_onnx

output_onnx_path = package_path / "onnx_model_.onnx"
quantized_onnx_path = package_path / "onnx_model_quantized.onnx"

final_path = install_and_export_sam_onnx(
    checkpoint=checkpoint,
    output_onnx=output_onnx_path,
    model_type="vit_h",
    return_single_mask=True,
    opset=17,
    quantize_out=quantized_onnx_path,  # here I did op_types_to_quantize=["MatMul", "Gemm"], which reduces quantization highly;
    # to mazimize quantization, do optmize_model = True instead
    gelu_approximate=False,
    use_stability_score=False,
    return_extra_metrics=False,
)
print("ONNX model saved to:", final_path)

[Info] Installing missing package 'onnxruntime-tools'...
[Info] Loading SAM model from checkpoint...


/opt/anaconda3/envs/section_identification/lib/python3.9/site-packages/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.lo

[Info] Exporting ONNX model to '/Users/fredericoaraujo/Documents/section_identification/onnx_model_.onnx'...


[Info] ONNX export completed: /Users/fredericoaraujo/Documents/section_identification/onnx_model_.onnx
[Info] Quantizing model => '/Users/fredericoaraujo/Documents/section_identification/onnx_model_quantized.onnx'...
[Info] Quantization completed.
[Info] Checking exported model with onnxruntime (CPU)...
[Success] Model 'onnx_model_quantized.onnx' runs successfully with ONNXRuntime.
ONNX model saved to: /Users/fredericoaraujo/Documents/section_identification/onnx_model_quantized.onnx
